# Structure
## Introduction
- What is Connect 4?
- (Fun) Facts
- Widget: Play using Bitbully Widget!

## Setup RL Problem
### 1. Environment
### 1.1. Bitboards
### 1.2. Basic Operations
     - Set stones
     - Switch players
     - Detect Wins
### 1.2. Visualization Widget
### 1.2. Advantages
### 2. State
#### What defines the state?

### 3. Agent
#### 3.1 N-tuple systems
#### 3.2 Tanh
#### 3.3 Value Function
#### 3.4 One set of LUTs per Player
#### 3.4 Online Network & Target Network

### 4. Policy
- Epsilon-Greedy

### 5. Actions

### 6. State Transition
- deterministic

### 7. Rewards
- +1, 0, -1

### 8. Update
- TD($\lambda$)
- Gradient Descent
- Adam-Optimizer



## 1. Environment

The **environment** is the Connect-4 game itself. It is fully deterministic: given a state and an action, the next state is uniquely determined — no stochasticity (unlike, e.g., Backgammon or card games).

The environment is implemented as `BoardBatch` — a vectorized, PyTorch-native board representation that can simulate **thousands of games in parallel** on CPU or GPU.

```
┌─────────────────────────────────────────────────────────────┐
│  BoardBatch (batch of B games)                              │
│                                                             │
│   all_tokens    : [B] int64  — every piece on the board     │
│   active_tokens : [B] int64  — only the current player's    │
│   moves_left    : [B] int16  — tokens remaining (max 42)    │
└─────────────────────────────────────────────────────────────┘
```

The opponent's tokens are never stored explicitly — they are derived on-the-fly as:

$$\text{opponent} = \texttt{active\_tokens} \oplus \texttt{all\_tokens}$$

### 1.1 Bitboards

Each board position is encoded as a **single 64-bit integer**. The 7 columns each occupy 9 bits (6 playable rows + 3 sentinel bits):

```
Bit indices in a uint64 (9 bits per column, 7 columns):

         C0    C1    C2    C3    C4    C5    C6
       ┌─────┬─────┬─────┬─────┬─────┬─────┬─────┐
  S2   │  8  │ 17  │ 26  │ 35  │ 44  │ 53  │ 62  │  ← sentinel
  S1   │  7  │ 16  │ 25  │ 34  │ 43  │ 52  │ 61  │  ← sentinel
  S0   │  6  │ 15  │ 24  │ 33  │ 42  │ 51  │ 60  │  ← sentinel
       ├─────┼─────┼─────┼─────┼─────┼─────┼─────┤
  R5   │  5  │ 14  │ 23  │ 32  │ 41  │ 50  │ 59  │  ← top row
  R4   │  4  │ 13  │ 22  │ 31  │ 40  │ 49  │ 58  │
  R3   │  3  │ 12  │ 21  │ 30  │ 39  │ 48  │ 57  │
  R2   │  2  │ 11  │ 20  │ 29  │ 38  │ 47  │ 56  │
  R1   │  1  │ 10  │ 19  │ 28  │ 37  │ 46  │ 55  │
  R0   │  0  │  9  │ 18  │ 27  │ 36  │ 45  │ 54  │  ← bottom row
       └─────┴─────┴─────┴─────┴─────┴─────┴─────┘
```

**Why sentinels?** Legal move generation uses an addition trick. Adding `BB_BOTTOM_ROW` to `all_tokens` propagates carry bits upward through a column. The 3 sentinel bits absorb carry from a full column, preventing it from overflowing into the next column.

$$\text{legal\_moves} = (\texttt{all\_tokens} + \texttt{BB\_BOTTOM\_ROW}) \mathbin{\&} \texttt{BB\_ALL\_LEGAL}$$

In [ ]:
import sys, os

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(".")), "src"))

import torch
from techdays26.torch_board import BoardBatch

# Create a batch of 1 empty board
board = BoardBatch.empty(1, device="cpu")

print("=== Empty board ===")
print(
    f"  all_tokens    = {int(board.all_tokens[0]):#066b}  ({int(board.all_tokens[0])})"
)
print(
    f"  active_tokens = {int(board.active_tokens[0]):#066b}  ({int(board.active_tokens[0])})"
)
print(f"  moves_left    = {int(board.moves_left[0])}")

# Drop Yellow's token into column 3 (center, bottom row → bit 27)
board.play_columns(torch.tensor([3]))

print("\n=== After Yellow plays column 3 ===")
print(f"  all_tokens    = {int(board.all_tokens[0]):#066b}")
print(f"  active_tokens = {int(board.active_tokens[0]):#066b}")
print(f"  moves_left    = {int(board.moves_left[0])}")

bit27_set = bool((int(board.all_tokens[0]) >> 27) & 1)
print(f"\n  Bit 27 (col 3, row 0) set in all_tokens? {bit27_set}  ✓")
print(f"  Active player now: {int(board.active_player()[0])}  (1=Yellow, 2=Red)")

### 1.2 Basic Operations

Three fundamental operations drive the game loop:

#### Set stones — `play_columns` / `play_masks`

Placing a token uses the same carry-propagation trick as legal move generation.  
`play_columns(cols)` finds the lowest empty bit in the chosen column and sets it in `all_tokens`.  
`play_masks(mv)` accepts a one-hot bitboard directly (used during tree search for speed).

In [ ]:
import torch
from techdays26.torch_board import BoardBatch

board = BoardBatch.empty(1, device="cpu")

# Drop three tokens into column 3 (stacking vertically: bits 27, 28, 29)
for col in [3, 3, 3]:
    board.play_columns(torch.tensor([col]))

# Decode which bits are set
bits_set = [i for i in range(64) if (int(board.all_tokens[0]) >> i) & 1]
print(f"Bits set in all_tokens after 3 tokens in col 3: {bits_set}")
print(f"  bit 27 = col 3, row 0  (bottom)")
print(f"  bit 28 = col 3, row 1")
print(f"  bit 29 = col 3, row 2")
print(f"\nmoves_left = {int(board.moves_left[0])}  (42 - 3 = 39)")

#### Switch players — a single XOR

After every move `play_columns` / `play_masks` calls:

```python
active_tokens ^= all_tokens
```

This is a single XOR instruction. Because `all_tokens` now includes the token just placed, XOR-ing with the *previous* `active_tokens` yields the **opponent's** pieces — which become the new `active_tokens` for the next turn. No copy, no extra field.

In [ ]:
board = BoardBatch.empty(1, device="cpu")

print("Turn | Player | active_tokens (bin, bits 27-31 only)")
print("-" * 55)
for turn, col in enumerate([3, 3, 3, 3]):
    player = int(board.active_player()[0])
    name = "Yellow" if player == 1 else "Red   "
    # Show bits 27..31 (col 3, rows 0..4) for compactness
    snippet = (int(board.active_tokens[0]) >> 27) & 0b11111
    print(f"  {turn}  | {name} | ...{snippet:05b}...  (bits 27-31 of col 3)")
    board.play_columns(torch.tensor([col]))

print(
    "\nFinal active_player:",
    int(board.active_player()[0]),
    "  (Yellow has played 2 tokens, Red 2 tokens)",
)

#### Detect wins — shift-AND in 4 directions

`has_win()` checks whether the player who just moved has four-in-a-row. It uses the **shift-AND** technique: shift a bitboard by the stride for a given direction, AND with the original. If two such shifted-AND results overlap, four consecutive bits are set.

| Direction  | Bit stride |
|------------|-----------|
| Horizontal | 9 (column offset) |
| Vertical   | 1 |
| Diagonal ↗ | 10 (col + 1) |
| Diagonal ↘ | 8  (col − 1) |

```python
# Vertical example (stride = 1):
x = y & (y << 2)       # pairs of 2 in the same column
win |= (x & (x << 1))  # 4 in a row
```

The entire win check runs in **O(1)** — no loop over cells.

In [ ]:
board = BoardBatch.empty(1, device="cpu")

# Yellow plays cols 0,1,2,3  (horizontal bottom row win)
# Red plays col 6 between each Yellow move to stay out of the way
move_sequence = [0, 6, 1, 6, 2, 6, 3]  # Yellow wins on last move

for i, col in enumerate(move_sequence):
    player = "Yellow" if i % 2 == 0 else "Red"
    board.play_columns(torch.tensor([col]))
    win = bool(board.has_win()[0])
    if win:
        print(f"Move {i + 1}: {player} plays col {col}  →  WIN detected!  ✓")
    else:
        print(f"Move {i + 1}: {player} plays col {col}  →  no win yet")

print(f"\nreward() = {float(board.reward()[0]):.1f}  (+1.0 = Yellow wins)")

### 1.3 Visualization Widget

The `BitboardVisualizer` renders the raw bitboard as an interactive Connect-4 board. Use it to build positions manually and inspect overlays (legal moves, winning threats, bit index map, win-check steps).

> See **[`lab2/bitboard_demo.ipynb`](bitboard_demo.ipynb)** for a full walkthrough of all overlays.

In [ ]:
%matplotlib ipympl
from techdays26.gui_bitboard import BitboardVisualizer

# Launch an interactive board — click columns to play, select overlay from the dropdown
vis = BitboardVisualizer()
vis.show()

### 1.4 Advantages of the Bitboard Representation

| Property | Benefit |
|---|---|
| **Compact** | Entire game state = 2 × int64 + 1 × int16 — fits in a few registers |
| **Vectorized** | `BoardBatch` stores a whole batch `[B]` of boards as three flat tensors; all operations are elementwise PyTorch ops |
| **GPU-ready** | Move `.to("cuda")` once; all subsequent ops run on the GPU without any Python loop |
| **O(1) win check** | Four shift-AND operations cover all four directions simultaneously |
| **No copies for player swap** | `active_tokens ^= all_tokens` — one instruction, in-place |
| **Scalable** | Training uses `B = 20 000` parallel games; the bitboard layout makes this practical |

## Future Work
- Deeper Networks?
- CNNs?
- Own Step-size Adaptation?